In [ ]:
!nvidia-smi

In [ ]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

In [ ]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate

In [ ]:
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
import pandas as pd

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

import nltk
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
import torch

nltk.download("punkt")

In [ ]:
device= "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

In [ ]:
!pip install -U transformers datasets evaluate sentencepiece

from transformers import AutoTokenizer

model_ckpt = "google/pegasus-cnn-dailymail"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

print("Tokenizer loaded successfully!")

In [ ]:
from transformers import AutoTokenizer

model_ckpt = "google/pegasus-xsum"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

print("Tokenizer loaded successfully!")

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
model_pegasus= AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

In [ ]:
!pip install -U datasets

In [ ]:
from datasets import load_dataset

dataset_samsum = load_dataset("knkarthick/samsum")
print(dataset_samsum)

In [ ]:
dataset_samsum["train"]["dialogue"][1]

In [ ]:
dataset_samsum["train"]["summary"][1]

In [ ]:
split_lengths= [len(dataset_samsum[split])for split in dataset_samsum]

print(f"Split lengths: {split_lengths}")
print(f"Features: {dataset_samsum['train'].column_names}")
print("\nDialogue: ")


print(dataset_samsum["test"][1]["dialogue"])

print("\nSummary: ")
print(dataset_samsum["test"][1]["summary"])

In [ ]:
def convert_examples_to_features(batch):
    inputs = tokenizer(
        batch["dialogue"],
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    targets = tokenizer(
        text_target=batch["summary"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    inputs["labels"] = targets["input_ids"]
    return inputs

In [ ]:
dataset_samsum_pt= dataset_samsum.map(convert_examples_to_features, batched= True)

In [ ]:
dataset_samsum_pt["train"]

In [ ]:
dataset_samsum_pt["train"]["input_ids"][1]

In [ ]:
dataset_samsum_pt["train"]["attention_mask"][1]

In [ ]:
dataset_samsum_pt["train"]["labels"][1]

In [ ]:
# Training
from transformers import DataCollatorForSeq2Seq

seq2seq_data_collator= DataCollatorForSeq2Seq(tokenizer, model= model_pegasus)

In [ ]:
from transformers import TrainingArguments, Trainer

trainer_args = TrainingArguments(
    output_dir="pegasus-samsum",
    num_train_epochs=1,
    warmup_steps=500,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=1000000,
    gradient_accumulation_steps=16
)

In [ ]:
trainer = Trainer(
    model=model_pegasus,
    args=trainer_args,
    processing_class=tokenizer,
    data_collator=seq2seq_data_collator,
    train_dataset=dataset_samsum_pt["test"],
    eval_dataset=dataset_samsum_pt["validation"]
)

In [ ]:
trainer.train()

In [ ]:
# Evaluation

def generate_batch_sized_chunks(list_of_elements, batch_size):
    "split the dataset into smaller batches that we can process simultaneously"
    "Yield successive batch-sized chunks from list."
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i:i + batch_size]


def calculate_metric_on_test_ds(dataset, metric, model, tokenizer,
                                batch_size=16, device=device,
                                column_text="dialogue",
                                column_summary="summary"):
    article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
    target_batches = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))

    for article_batch, target_batch in tqdm(
        zip(article_batches, target_batches), total=len(article_batches)
    ):

        inputs = tokenizer(
            article_batch,
            max_length=1024,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        summaries = model.generate(
            input_ids=inputs["input_ids"].to(device),
            attention_mask=inputs["attention_mask"].to(device),
            length_penalty=0.8,
            num_beams=8,
            max_length=128
        )

        "parameter for length penalty ensures that the model does not generate sequences that are so long"

        # finally we decode the generated texts,
        # replace the token, and add the decoded texts with the references to the metrics

        decoded_summaries = [
            tokenizer.decode(
                s,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True
            )
            for s in summaries
        ]
        decoded_summaries = [d.replace("", " ") for d in decoded_summaries]

        metric.add_batch(predictions=decoded_summaries, references=target_batch)

    # finaly compute and return the Rouge scores
    score = metric.compute()
    return score

In [ ]:
!pip install evaluate

In [ ]:
!pip install rouge_score

In [ ]:
import evaluate

rouge_metric = evaluate.load("rouge")

In [29]:
import evaluate

rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
rouge_metric = evaluate.load("rouge")

In [ ]:
score= calculate_metric_on_test_ds(
    dataset_samsum["test"][0:5], rouge_metric, trainer.model, tokenizer, batch_size=2, column_text="dialogue", column_summary="summary"
)

rogue_dict= dict((rn, score[rn].mid.fmeasure) for rn in rouge_names)

pd.DataFrame(rogue_dict, index=[f"pegasus"])


In [ ]:
# save the model
model_pegasus.save_pretrained("pegasus-samsum-model")

In [ ]:
# save tokenizer

tokenizer.save_pretrained("tokenizer")